# ORM vs Raw SQL — Benchmark Analysis

**Study:** Évaluation de l'impact des couches ORM sur la latence et la consommation de ressources d'une API  
**Author:** Nejma Moualhi — FISA Informatique A4, CESI  
**Run:** `run_20260331_115700`

---

## Setup

In [ ]:
import json
import os
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from pathlib import Path

# ── Config ──────────────────────────────────────────────────────────────────
RUN_DIR = Path("results/run_20260331_115700")

COLORS = {"sql": "#2196F3", "orm": "#FF9800"}   # blue = SQL, orange = Prisma
THRESHOLD_PCT = 20   # acceptable ORM overhead (%)

SCENARIO_LABELS = {
    "01-smoke":           "Smoke\n(1 VU)",
    "02-crud":            "CRUD\n(50 VUs)",
    "03-read-heavy":      "Read-heavy\n(100 VUs)",
    "04-complex-queries": "Complex\nqueries (50 VUs)",
    "05-stress":          "Stress\n(200 VUs)",
}

plt.rcParams.update({
    "figure.dpi": 130,
    "font.family": "sans-serif",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
print("Setup OK — run dir:", RUN_DIR)

## 1. Load all results

In [ ]:
def load_summary(path):
    """Load a k6 summary JSON and return flat metric dict."""
    with open(path) as f:
        d = json.load(f)
    metrics = d.get("metrics", {})
    result = {}
    for key, val in metrics.items():
        if isinstance(val, dict):
            result[key] = val
    return result

def get_metric(metrics, key, stat="p(95)"):
    m = metrics.get(key, {})
    return m.get(stat, m.get("value", 0)) or 0

# ── Parse all summary files ──────────────────────────────────────────────────
records = []
for path in sorted(RUN_DIR.glob("*_summary.json")):
    name = path.stem.replace("_summary", "")
    parts = name.rsplit("_", 1)
    if len(parts) != 2:
        continue
    scenario, impl = parts
    m = load_summary(path)

    dur = m.get("http_req_duration", {})
    records.append({
        "scenario": scenario,
        "impl":     impl,
        "avg_ms":   dur.get("avg", 0),
        "med_ms":   dur.get("med", 0),
        "p90_ms":   dur.get("p(90)", 0),
        "p95_ms":   dur.get("p(95)", 0),
        "max_ms":   dur.get("max", 0),
        "throughput": m.get("http_reqs", {}).get("rate", 0),
        "total_reqs": m.get("http_reqs", {}).get("count", 0),
        "error_rate": (m.get("http_req_failed", {}).get("value", 0) or 0) * 100,
        "_metrics":   m,   # keep raw for per-op breakdown
    })

df = pd.DataFrame(records)
df["label"] = df["scenario"].map(SCENARIO_LABELS)

print(f"Loaded {len(df)} result sets across {df['scenario'].nunique()} scenarios")
df[["scenario", "impl", "avg_ms", "p95_ms", "throughput", "error_rate"]].round(2)

## 2. P95 Latency — Overall Comparison

Primary metric per the study protocol. The **20% threshold** line marks when the ORM overhead becomes scientifically significant.

In [ ]:
scenarios = list(SCENARIO_LABELS.keys())
x = np.arange(len(scenarios))
w = 0.35

fig, ax = plt.subplots(figsize=(12, 5))

for i, impl in enumerate(["sql", "orm"]):
    sub = df[df.impl == impl].set_index("scenario")
    vals = [sub.loc[s, "p95_ms"] if s in sub.index else 0 for s in scenarios]
    bars = ax.bar(x + i*w - w/2, vals, width=w,
                  label="Raw SQL (pg)" if impl == "sql" else "Prisma ORM",
                  color=COLORS[impl], alpha=0.88, zorder=3)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 15,
                    f"{v:.0f}ms", ha="center", va="bottom", fontsize=8, fontweight="bold")

# 20% threshold band (only meaningful relative to SQL bar — shown as annotation)
ax.set_xticks(x)
ax.set_xticklabels([SCENARIO_LABELS[s] for s in scenarios], fontsize=9)
ax.set_ylabel("P95 Latency (ms)", fontsize=11)
ax.set_title("P95 Latency — Raw SQL vs Prisma ORM across all scenarios", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
ax.set_yscale("log")   # log scale because range spans 3.5ms → 2678ms
ax.set_ylabel("P95 Latency (ms) — log scale", fontsize=11)

# Annotate the big reversal
ax.annotate("← SQL saturates\nunder concurrent load",
            xy=(2 - w/2, 1695), xytext=(2.8, 1200),
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=8, color="gray")

plt.tight_layout()
plt.savefig(RUN_DIR / "chart_p95_overall.png", dpi=150)
plt.show()
print("Saved chart_p95_overall.png")

## 3. ORM Overhead (%) vs the 20% Threshold

Positive = ORM is slower. Negative = ORM is faster. The **red dashed line at +20%** is the acceptable threshold defined in the study protocol.

In [ ]:
pivot = df.pivot(index="scenario", columns="impl", values="p95_ms")
pivot["delta_pct"] = (pivot["orm"] - pivot["sql"]) / pivot["sql"] * 100
pivot = pivot.reindex(scenarios)

fig, ax = plt.subplots(figsize=(10, 5))

bar_colors = ["#e53935" if v > THRESHOLD_PCT else ("#43a047" if v < 0 else "#fb8c00")
              for v in pivot["delta_pct"]]

bars = ax.bar([SCENARIO_LABELS[s] for s in scenarios],
              pivot["delta_pct"], color=bar_colors, alpha=0.88, zorder=3)

for bar, v in zip(bars, pivot["delta_pct"]):
    va = "bottom" if v >= 0 else "top"
    offset = 2 if v >= 0 else -2
    ax.text(bar.get_x() + bar.get_width()/2, v + offset,
            f"{v:+.1f}%", ha="center", va=va, fontsize=9, fontweight="bold")

ax.axhline(THRESHOLD_PCT,  color="#e53935", linestyle="--", linewidth=1.5,
           label=f"+{THRESHOLD_PCT}% acceptable threshold")
ax.axhline(0, color="black", linewidth=0.8)

ax.set_ylabel("P95 ORM overhead vs SQL (%)\nNegative = ORM faster", fontsize=10)
ax.set_title("ORM overhead per scenario — does Prisma exceed the 20% threshold?",
             fontsize=12, fontweight="bold")
ax.legend(fontsize=9)
ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)

# Legend patches
patches = [
    mpatches.Patch(color="#e53935", label="ORM exceeds threshold"),
    mpatches.Patch(color="#fb8c00", label="ORM slower, within threshold"),
    mpatches.Patch(color="#43a047", label="ORM faster than SQL"),
]
ax.legend(handles=patches, fontsize=9)

plt.tight_layout()
plt.savefig(RUN_DIR / "chart_overhead_pct.png", dpi=150)
plt.show()

## 4. Throughput (requests/second)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))

for i, impl in enumerate(["sql", "orm"]):
    sub = df[df.impl == impl].set_index("scenario")
    vals = [sub.loc[s, "throughput"] if s in sub.index else 0 for s in scenarios]
    bars = ax.bar(x + i*w - w/2, vals, width=w,
                  label="Raw SQL (pg)" if impl == "sql" else "Prisma ORM",
                  color=COLORS[impl], alpha=0.88, zorder=3)
    for bar, v in zip(bars, vals):
        if v > 0:
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 3,
                    f"{v:.0f}", ha="center", va="bottom", fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels([SCENARIO_LABELS[s] for s in scenarios], fontsize=9)
ax.set_ylabel("Throughput (req/s)", fontsize=11)
ax.set_title("Throughput — Raw SQL vs Prisma ORM", fontsize=13, fontweight="bold")
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)

plt.tight_layout()
plt.savefig(RUN_DIR / "chart_throughput.png", dpi=150)
plt.show()

## 5. Per-Operation Latency Breakdown

Scenarios 03 (read-heavy) and 04 (complex queries) have tagged operations. This shows where each implementation struggles or excels.

In [ ]:
def extract_op_p95(metrics):
    """Extract per-operation P95 from tagged http_req_duration metrics."""
    ops = {}
    for key, val in metrics.items():
        if key.startswith("http_req_duration{op:"):
            op = key.split("op:")[1].rstrip("}")
            ops[op] = val.get("p(95)", 0)
    return ops

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scenario in zip(axes, ["03-read-heavy", "04-complex-queries"]):
    op_data = {}
    for impl in ["sql", "orm"]:
        row = df[(df.scenario == scenario) & (df.impl == impl)].iloc[0]
        op_data[impl] = extract_op_p95(row["_metrics"])

    # Union of all ops
    all_ops = sorted(set(list(op_data["sql"].keys()) + list(op_data["orm"].keys())))
    xo = np.arange(len(all_ops))

    for i, impl in enumerate(["sql", "orm"]):
        vals = [op_data[impl].get(op, 0) for op in all_ops]
        bars = ax.bar(xo + i*0.35 - 0.175, vals, width=0.35,
                      label="Raw SQL" if impl == "sql" else "Prisma ORM",
                      color=COLORS[impl], alpha=0.88, zorder=3)
        for bar, v in zip(bars, vals):
            if v > 0:
                ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                        f"{v:.0f}ms", ha="center", va="bottom", fontsize=7.5)

    ax.set_xticks(xo)
    op_labels = [op.replace("_", "\n") for op in all_ops]
    ax.set_xticklabels(op_labels, fontsize=8)
    ax.set_ylabel("P95 Latency (ms)", fontsize=10)
    ax.set_title(f"{SCENARIO_LABELS[scenario].replace(chr(10), ' ')} — per-operation P95",
                 fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4, zorder=0)

plt.suptitle("Per-operation breakdown: where does each implementation struggle?",
             fontsize=12, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig(RUN_DIR / "chart_per_op_breakdown.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Latency Percentile Profiles (avg / P90 / P95)

Shows the shape of the latency distribution — a wide spread between avg and P95 indicates high variance (tail latency problem).

In [ ]:
# Focus on the 3 core scenarios (protocol §10)
core = ["02-crud", "03-read-heavy", "04-complex-queries"]

fig, axes = plt.subplots(1, 3, figsize=(14, 5), sharey=False)

for ax, scenario in zip(axes, core):
    for impl in ["sql", "orm"]:
        row = df[(df.scenario == scenario) & (df.impl == impl)].iloc[0]
        percentiles = ["avg_ms", "p90_ms", "p95_ms"]
        labels_p    = ["Avg", "P90", "P95"]
        vals = [row[p] for p in percentiles]
        ax.plot(labels_p, vals, marker="o", linewidth=2,
                color=COLORS[impl],
                label="Raw SQL" if impl == "sql" else "Prisma ORM")
        ax.fill_between(labels_p, vals, alpha=0.08, color=COLORS[impl])

    ax.set_title(SCENARIO_LABELS[scenario].replace("\n", " "), fontsize=11, fontweight="bold")
    ax.set_ylabel("Latency (ms)", fontsize=9)
    ax.yaxis.grid(True, linestyle="--", alpha=0.4)
    ax.legend(fontsize=9)

plt.suptitle("Latency percentile profiles — Avg / P90 / P95 (core scenarios)",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(RUN_DIR / "chart_percentile_profiles.png", dpi=150)
plt.show()

## 7. Code Complexity — Lines of Code

In [ ]:
loc_data = {
    "Component": [
        "Application code\n(src/)",
        "Schema /\nDB definition",
        "Total",
    ],
    "Raw SQL (pg)": [397, 67, 464],   # init.sql = 67 lines
    "Prisma ORM":   [338, 62, 400],   # schema.prisma = 62 lines
}
# Note: init.sql is shared but conceptually "belongs" to sql-api as its schema def

loc_df = pd.DataFrame(loc_data).set_index("Component")

fig, ax = plt.subplots(figsize=(8, 4.5))
xc = np.arange(len(loc_df))

for i, (col, color) in enumerate([("Raw SQL (pg)", COLORS["sql"]), ("Prisma ORM", COLORS["orm"])]):
    bars = ax.bar(xc + i*0.35 - 0.175, loc_df[col], width=0.35,
                  label=col, color=color, alpha=0.88, zorder=3)
    for bar, v in zip(bars, loc_df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
                str(v), ha="center", va="bottom", fontsize=10, fontweight="bold")

ax.set_xticks(xc)
ax.set_xticklabels(loc_df.index, fontsize=10)
ax.set_ylabel("Lines of Code", fontsize=11)
ax.set_title("Developer productivity — Lines of Code comparison", fontsize=12, fontweight="bold")
ax.legend(fontsize=10)
ax.yaxis.grid(True, linestyle="--", alpha=0.5, zorder=0)
ax.set_ylim(0, 550)

ax.annotate("Prisma saves ~15%\nin application code\nbut similar total",
            xy=(0.63, 400), xytext=(1.2, 450),
            arrowprops=dict(arrowstyle="->", color="gray"),
            fontsize=8.5, color="gray")

plt.tight_layout()
plt.savefig(RUN_DIR / "chart_loc.png", dpi=150)
plt.show()

## 8. Summary Table — All Metrics

In [ ]:
pivot_full = df.pivot_table(
    index="scenario", columns="impl",
    values=["avg_ms", "p95_ms", "throughput", "error_rate", "total_reqs"]
).round(2)

# Compute delta
p95_pivot = df.pivot(index="scenario", columns="impl", values="p95_ms")
p95_pivot["delta_pct"] = ((p95_pivot["orm"] - p95_pivot["sql"]) / p95_pivot["sql"] * 100).round(1)
p95_pivot["verdict"] = p95_pivot["delta_pct"].apply(
    lambda d: "⚠ Exceeds threshold" if d > 20 else ("✓ ORM faster" if d < 0 else "✓ Within threshold")
)

print("=== P95 Latency Comparison ===")
print(p95_pivot.reindex(scenarios).to_string())
print()
print("=== Full Metrics ===")
display(pivot_full.reindex(scenarios))

## 9. Key Findings & Interpretation

### What the data shows

| Scenario | Finding |
|---|---|
| **Smoke (1 VU)** | Raw SQL is 2× faster at low load — ORM startup overhead visible (P95: 3.5ms vs 6.7ms) |
| **CRUD (50 VUs)** | ORM overhead is **+14.5%** — within the 20% acceptable threshold |
| **Read-heavy (100 VUs)** | **ORM wins by 87%** — sql-api saturates (P95: 1695ms vs 220ms) |
| **Complex queries (50 VUs)** | **ORM wins by 64%** — Prisma's split-query approach outperforms `json_agg` |
| **Stress (200 VUs)** | **ORM wins by 92%** — sql-api near saturation (P95: 2678ms vs 220ms) |

### Why did raw SQL slow down under load?

The `sql-api` uses **single complex queries with `json_agg + GROUP BY`** to fetch posts with their tags in one round-trip:
```sql
SELECT p.*, json_agg(t.*) AS tags FROM posts p
JOIN users u ON ... LEFT JOIN post_tags pt ON ... LEFT JOIN tags t ON ...
GROUP BY p.id, u.id
LIMIT 20
```
Under 100 concurrent users, this query:
- Holds a connection **longer** per request (aggregation is CPU-heavy on PostgreSQL)
- Creates **contention** in the connection pool (max=20)
- Each waiting VU adds to the queue → latency compounds

The `orm-api` (Prisma) uses **separate, simpler queries**:
1. `SELECT * FROM posts WHERE ... LIMIT 20`
2. `SELECT * FROM tags WHERE post_id IN (...)`

Each query is short, releases the connection faster → better concurrency.

### Hypothesis validation

| Hypothesis | Result |
|---|---|
| H1: SQL faster on complex joins | **Refuted** — ORM is faster under concurrency |
| H2: Gap negligible on simple CRUD | **Confirmed** — +14.5% P95 on CRUD, within 20% threshold |
| H3: ORM overhead increases with concurrency | **Refuted** — the opposite is true |
| H4: Prisma has less application code | **Partially confirmed** — 15% less src LOC, but similar total |

### Important nuance for the report

> The raw SQL implementation used a **single-query aggregation strategy** (`json_agg`). A different SQL implementation using multiple simple queries would likely perform similarly to Prisma. The performance difference observed is therefore **not between SQL and ORM per se**, but between **two query strategies**: aggregated single-query (sql-api) vs. split multi-query (Prisma). This nuance must be stated clearly in the report conclusions.

## 10. Export all charts

In [ ]:
charts = list(RUN_DIR.glob("chart_*.png"))
print(f"{len(charts)} charts saved in {RUN_DIR}:")
for c in sorted(charts):
    size_kb = c.stat().st_size // 1024
    print(f"  {c.name} ({size_kb} KB)")